In [ ]:

import pandas as pd
import pandas as pd
import numpy as np
import json
from pathlib import Path
import joblib
from sentence_transformers import SentenceTransformer
import faiss
from keras.models import load_model
from transformers import DistilBertTokenizer, TFDistilBertModel
import tensorflow as tf
resume_dataset=pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Projet MASTER1/cleaned_combined_resume_final.csv')
offer_dataset=pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Projet MASTER1/cleaned_combined_offer_final.csv')

In [ ]:
#Lecture et affichage Information dataframme
def read_data(data,column_name):
    print("\n __Aperçu des données :")
    print(data.head(3))
    # Afficher les informations sur le DataFrame
    print("\n __Informations sur les données :")
    print(data.info())
    print(" \n __Taille : \n",data.shape)
    print("\n __Information sur les Categories: \n",data[column_name].value_counts().reset_index())


### **Echantillonage**

In [ ]:
import hashlib

def generate_id(text):
    return hashlib.md5(text.encode()).hexdigest()[:12]

In [ ]:
def balanced_sampling(dataset, total_samples, threshold_min, threshold_rare, category_column='Category'):
    """
    Génère un échantillon équilibré selon les seuils spécifiés.
    """
    # Calculer la fréquence des catégories
    category_counts = dataset[category_column].value_counts().reset_index()
    category_counts.columns = [category_column, 'Count']

    # Initialiser un DataFrame pour l'échantillon final
    sampled_data = pd.DataFrame()

    # Étape 1 : Prendre TOUTES les catégories rares
    rare_categories = category_counts[category_counts['Count'] < threshold_rare][category_column]
    for category in rare_categories:
        category_data = dataset[dataset[category_column] == category]
        sampled_data = pd.concat([sampled_data, category_data])

    # Étape 2 : Échantillonner les catégories fréquentes
    frequent_categories = category_counts[category_counts['Count'] >= threshold_rare][category_column]
    remaining_samples = total_samples - len(sampled_data)

    for category in frequent_categories:
        category_data = dataset[dataset[category_column] == category]
        proportion = len(category_data) / len(dataset)
        target_samples = max(int(proportion * remaining_samples), threshold_min)
        n_samples = min(target_samples, len(category_data))

        sampled_category = category_data.sample(n=n_samples, random_state=42)
        sampled_data = pd.concat([sampled_data, sampled_category])

    # Étape 3 : Ajuster si le total dépasse total_samples
    if len(sampled_data) > total_samples:
        excess = len(sampled_data) - total_samples
        large_categories = sampled_data[category_column].value_counts().index.tolist()
        to_remove = pd.DataFrame()

        for cat in large_categories:
            if excess <= 0:
                break
            cat_data = sampled_data[sampled_data[category_column] == cat]
            # Ne pas descendre en dessous du threshold_min pour les catégories fréquentes
            min_threshold = threshold_min if cat in frequent_categories.tolist() else 0
            n_remove = min(excess, len(cat_data) - min_threshold)
            if n_remove > 0:
                removed = cat_data.sample(n=n_remove, random_state=42)
                to_remove = pd.concat([to_remove, removed])
                excess -= n_remove

        sampled_data = sampled_data.drop(to_remove.index)

    return sampled_data.reset_index(drop=True)

In [ ]:
def load_data_sampling():
    """
    Charge les données originales et applique l'échantillonnage équilibré
    Retourne les datasets échantillonnés pour CVs et offres
    """
    # Charger les datasets originaux
    resume_dataset = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Projet MASTER1/cleaned_combined_resume_final.csv')
    offer_dataset = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Projet MASTER1/cleaned_combined_offer_final.csv')

    # Créer des échantillons équilibrés
    cv_dataset = balanced_sampling(resume_dataset, 1200, 35, 40)
    offer_dataset = balanced_sampling(offer_dataset, 800, 35, 50, 'Job Title_Category')

    # Nettoyage supplémentaire
    cv_dataset = cv_dataset.dropna(subset=['Resume']).reset_index(drop=True)
    offer_dataset = offer_dataset.dropna(subset=['Job Description']).reset_index(drop=True)

    # Création d'IDs persistants basés sur le contenu
    cv_dataset['cv_id'] = cv_dataset['Resume'].apply(generate_id)
    offer_dataset['offer_id'] = offer_dataset['Job Description'].apply(generate_id)

    # Création de mappings pour retrouver les index à partir des IDs
    cv_id_to_index = {id_: idx for idx, id_ in enumerate(cv_dataset['cv_id'])}
    offer_id_to_index = {id_: idx for idx, id_ in enumerate(offer_dataset['offer_id'])}

    print(f"Échantillon CVs créé: {len(cv_dataset)} éléments")
    print(f"Échantillon offres créé: {len(offer_dataset)} éléments")

    return cv_dataset, offer_dataset, cv_id_to_index, offer_id_to_index

##### Fonctions de Sauvegarde et Chargement des Échantillons

In [ ]:
# def save_sampled_data(cv_dataset, offer_dataset, cv_id_to_index, offer_id_to_index, save_dir):
#     """
#     Sauvegarde les données échantillonnées pour une utilisation ultérieure
#     """
#     save_dir = Path(save_dir)
#     save_dir.mkdir(exist_ok=True)

#     # Sauvegarder les datasets
#     cv_dataset.to_csv(save_dir / 'sampled_cv_metadata.csv', index=False)
#     offer_dataset.to_csv(save_dir / 'sampled_offer_metadata.csv', index=False)

#     # Sauvegarder les mappings
#     with open(save_dir / 'sampled_cv_id_mapping.json', 'w') as f:
#         json.dump(cv_id_to_index, f)

#     with open(save_dir / 'sampled_offer_id_mapping.json', 'w') as f:
#         json.dump(offer_id_to_index, f)

#     print("Données échantillonnées sauvegardées")

In [ ]:

# def load_sampled_data(save_dir):
#     """
#     Charge les données échantillonnées préalablement sauvegardées
#     """
#     save_dir = Path(save_dir)

#     # Charger les datasets
#     cv_dataset = pd.read_csv(save_dir / 'sampled_cv_metadata.csv')
#     offer_dataset = pd.read_csv(save_dir / 'sampled_offer_metadata.csv')

#     # Charger les mappings
#     with open(save_dir / 'sampled_cv_id_mapping.json', 'r') as f:
#         cv_id_to_index = json.load(f)

#     with open(save_dir / 'sampled_offer_id_mapping.json', 'r') as f:
#         offer_id_to_index = json.load(f)

#     print("Données échantillonnées chargées")

#     return cv_dataset, offer_dataset, cv_id_to_index, offer_id_to_index

### **Fonctions Principales**

#### Generation embedding et chargement data

In [ ]:
def generate_embeddings(cv_dataset, offer_dataset):
    """
    Génère les embeddings et les index FAISS pour les datasets fournis
    """
    # Charger le modèle SBERT
    sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

    # Génération des embeddings pour tous les CV
    print("Génération des embeddings pour les CV...")
    cv_texts = cv_dataset['Resume'].tolist()
    cv_embeddings = sbert_model.encode(cv_texts, convert_to_tensor=True, show_progress_bar=True)

    # Génération des embeddings pour toutes les offres
    print("Génération des embeddings pour les offres...")
    offer_texts = offer_dataset['Job Description'].tolist()
    offer_embeddings = sbert_model.encode(offer_texts, convert_to_tensor=True, show_progress_bar=True)

    # Conversion en numpy array pour FAISS
    cv_embeddings_np = cv_embeddings.cpu().numpy()
    offer_embeddings_np = offer_embeddings.cpu().numpy()

    # Normalisation des vecteurs pour la similarité cosinus
    faiss.normalize_L2(cv_embeddings_np)
    faiss.normalize_L2(offer_embeddings_np)

    # Création des index FAISS pour une recherche rapide
    dimension = cv_embeddings_np.shape[1]
    cv_index = faiss.IndexFlatIP(dimension)
    offer_index = faiss.IndexFlatIP(dimension)

    # Ajout des vecteurs aux index
    cv_index.add(cv_embeddings_np)
    offer_index.add(offer_embeddings_np)

    print("Embeddings et index FAISS générés avec succès")

    return cv_index, offer_index, cv_embeddings_np, offer_embeddings_np, sbert_model

In [ ]:
def load_matching_data(use_sampled_data=True):
    """
    Charge les données et index pour le matching sémantique
    Utilise soit les données complètes soit des échantillons selon le paramètre use_sampled_data
    """
    if use_sampled_data:
        # Charger les données échantillonnées
        cv_dataset, offer_dataset, cv_id_to_index, offer_id_to_index = load_data_sampling()

        # Générer les embeddings
        cv_index, offer_index, cv_embeddings, offer_embeddings, sbert_model = generate_embeddings(
            cv_dataset, offer_dataset)

        return (cv_dataset, offer_dataset, cv_index, offer_index,
                cv_embeddings, offer_embeddings, cv_id_to_index,
                offer_id_to_index, sbert_model)

    else:
      return None
        # # Utiliser les données complètes (code original)
        # matching_dir = Path('/content/drive/MyDrive/Colab Notebooks/Projet MASTER1/matching_data')

        # # Charger les datasets
        # cv_dataset = pd.read_csv(matching_dir / 'cv_metadata.csv')
        # offer_dataset = pd.read_csv(matching_dir / 'offer_metadata.csv')

        # # Charger les index FAISS
        # cv_index = faiss.read_index(str(matching_dir / 'cv_index.faiss'))
        # offer_index = faiss.read_index(str(matching_dir / 'offer_index.faiss'))

        # # Charger les embeddings
        # cv_embeddings = np.load(matching_dir / 'cv_embeddings.npy')
        # offer_embeddings = np.load(matching_dir / 'offer_embeddings.npy')

        # # Charger les mappings ID->index
        # with open(matching_dir / 'cv_id_mapping.json', 'r') as f:
        #     cv_id_to_index = json.load(f)

        # with open(matching_dir / 'offer_id_mapping.json', 'r') as f:
        #     offer_id_to_index = json.load(f)

        # # Charger le modèle SBERT pour le matching
        # sbert_model = SentenceTransformer('all-MiniLM-L6-v2')

        # return (cv_dataset, offer_dataset, cv_index, offer_index,
        #         cv_embeddings, offer_embeddings, cv_id_to_index,
        #         offer_id_to_index, sbert_model)

#### Classification



In [ ]:
def load_classification_models():
    """Charge les modèles de classification pré-entraînés"""
    models_dir = Path('/content/drive/MyDrive/Colab Notebooks/Projet MASTER1/saved_model')

    # Vérifier que le dossier existe
    if not models_dir.exists():
        raise FileNotFoundError(f"Le dossier {models_dir} n'existe pas")

    # Liste des fichiers requis
    required_files = [
        'CNN_final_model_classifier_resume.keras',
        'CNN_final_model_classifier_offer.keras',
        'label_encoder_resume.joblib',
        'label_encoder_offer.joblib'
    ]

    for file in required_files:
        if not (models_dir / file).exists():
            raise FileNotFoundError(f"Fichier manquant: {file}")

    # Chargement des modèles
    cv_classifier = load_model(models_dir / 'CNN_final_model_classifier_resume.keras')
    offer_classifier = load_model(models_dir / 'CNN_final_model_classifier_offer.keras')
    cv_le = joblib.load(models_dir / 'label_encoder_resume.joblib')
    offer_le = joblib.load(models_dir / 'label_encoder_offer.joblib')
    tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
    distilbert_model = TFDistilBertModel.from_pretrained('distilbert-base-uncased', from_pt=True)

    return cv_classifier, offer_classifier, cv_le, offer_le, tokenizer, distilbert_model

def load_high_confidence_classes(confidence_threshold=0.7):
    """
    Détermine les classes avec f1-score > seuil basé sur l'historique d'entraînement
    """
    try:
        # Charger les métriques d'évaluation
        cv_class_acc = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Projet MASTER1/saved_model/resume_class_metrics.csv')
        offer_class_acc = pd.read_csv('/content/drive/MyDrive/Colab Notebooks/Projet MASTER1/saved_model/offer_class_metrics.csv')

        # Filtrer les classes avec f1-score > threshold
        high_conf_cv_classes = cv_class_acc[cv_class_acc['f1_score'] > confidence_threshold]['class_name'].tolist()
        high_conf_offer_classes = offer_class_acc[offer_class_acc['f1_score'] > confidence_threshold]['class_name'].tolist()

    except:
        # Fallback: utiliser toutes les classes si les fichiers ne sont pas disponibles
        print("Fichiers de métriques non trouvés, utilisation de toutes les classes")
        cv_classifier, _, cv_le, _, _, _ = load_classification_models()
        high_conf_cv_classes = cv_le.classes_.tolist()

        _, offer_classifier, _, offer_le, _, _ = load_classification_models()
        high_conf_offer_classes = offer_le.classes_.tolist()

    return high_conf_cv_classes, high_conf_offer_classes

##### **Mapping Class**

In [ ]:
def create_manual_class_mapping():
    """
    Crée un mapping manuel entre les classes de CVs et d'offres
    """
    manual_mapping = {
        # IT/Technologie - Correspondances exactes ou très similaires
        'Data Scientist': 'Data Scientist',
        'Software Developer': 'Software Developer',
        'Data Engineer': 'Data Engineer',
        'Web Developer': 'Web Developer',
        'DevOps/Cloud': 'Data Engineer',
        'Database Administrator': 'Database Administrator',
        'Network Administrator': 'Network Administrator',
        'Systems Administrator': 'Systems Administrator',
        'Security Analyst': 'Information Security Analyst',
        'Cyber Security Analyst': 'Information Security Analyst',
        'Information Security Analyst': 'Information Security Analyst',
        'Project Manager': 'Project Manager',
        'Software Engineer': 'Software Engineer',
        'AI/ML Specialist': 'Data Scientist',
        'Hardware Engineer': 'Hardware Engineer',
        'Data Analyst': 'Data Engineer',

        # Big Data & Cloud
        'Big Data Cloud Developer': 'Data Engineer',

        # Développeurs
        'Python Developer': 'Python / Java Developer',
        'Java Developer': 'Python / Java Developer',
        'Java Full Stack Developer': 'Python / Java Developer',
        'Python Developer,Software Developer': 'Python / Java Developer',
        'Java Developer,Software Developer': 'Python / Java Developer',
        'Front End Developer': 'Web Developer',
        'Full Stack Developer': 'Web Developer',
        'UI Developer': 'Web Developer',
        'Mobile Developer': 'Software Developer',
        'Java/J2Ee Developer': 'Python / Java Developer',
        'Senior Java Full Stack Developer': 'Python / Java Developer',
        'Senior Python Developer': 'Python / Java Developer',

        # Base de données
        'Oracle Database Administrator': 'Database Administrator',
        'SQL/SQL Server Database Administrator': 'Database Administrator',
        'Senior Database Administrator': 'Database Administrator',

        # Cybersécurité
        'Network Engineer': 'Network Administrator',

        # Management
        'It Manager': 'Other It Professional',
        'Consultant': 'Other It Professional',
        'Project Manager,Software Developer': 'Project Manager',

        # Catégories non-IT
        'Chef/Culinary': 'Chef/Culinary',
        'Mechanical Engineer': 'Mechanical Engineer',
        'Teacher/Professor': 'Teacher/Professor',
        'Legal Counsel': 'Legal Counsel',
        'HR Professional': 'HR Professional',
        'Financial Specialist': 'Financial Specialist',
        'Librarian/Archivist': 'Librarian/Archivist',
        'Research Scientist': 'Research Scientist',
        'Loan Officer': 'Loan Officer',
        'Bartender/Server': 'Bartender/Server',
        'Actuarial Specialist': 'Actuarial Specialist',
        'Counselor': 'Counselor',
        'Executive/C-Suite': 'Executive/C-Suite',

        # Catégories génériques
        'Other IT': 'Other It Professional',
        'Other No IT': 'Other No IT Professionnel',
        'Job Seeker': 'Other No IT Professionnel',
    }

    return manual_mapping

### **Classification et Matching hybrid**

In [ ]:
def classify_text(text, text_type, classifier, le, tokenizer, distilbert_model, confidence_threshold=0.7):
    """
    Classifie un texte (CV ou offre) et retourne la classe prédite et le score de confiance
    """
    # Tokenization
    inputs = tokenizer(text, truncation=True, padding=True, max_length=256, return_tensors="tf")

    # Génération des embeddings avec DistilBERT
    outputs = distilbert_model(inputs)
    embeddings = outputs.last_hidden_state

    # Prédiction avec le modèle CNN
    predictions = classifier.predict(embeddings)
    predicted_class_idx = np.argmax(predictions, axis=1)[0]
    confidence_score = np.max(predictions)

    # Décodage de la classe
    predicted_class = le.inverse_transform([predicted_class_idx])[0]

    # Vérification du seuil de confiance
    is_high_confidence = confidence_score >= confidence_threshold

    return predicted_class, confidence_score, is_high_confidence

def get_target_classes(source_class, source_type, manual_mapping):
    """
    Obtient les classes cibles basées sur le mapping manuel
    """
    target_classes = []

    if source_type == 'cv':
        # CV -> Offres: mapping direct
        if source_class in manual_mapping:
            target_classes.append(manual_mapping[source_class])
        else:
            # Si la classe n'est pas dans le mapping, utiliser la classe source comme fallback
            target_classes.append(source_class)
    else:
        # Offres -> CVs: mapping inverse
        target_classes = [cv_class for cv_class, offer_class in manual_mapping.items()
                         if offer_class == source_class]

        # Si aucun mapping inverse trouvé, utiliser la classe source comme fallback
        if not target_classes:
            target_classes.append(source_class)

    return target_classes

def filter_by_class(dataset, target_classes, id_to_index, embeddings, index):
    """
    Filtre les embeddings et indices par classe cible
    """
    # Trouver les IDs des documents dans les classes cibles
    target_ids = dataset[dataset.iloc[:, 1].isin(target_classes)].iloc[:, 0].tolist()

    # Trouver les indices FAISS correspondants
    target_indices = [id_to_index[doc_id] for doc_id in target_ids if doc_id in id_to_index]

    if not target_indices:
        return None, None

    # Créer un sous-ensemble d'embeddings
    target_embeddings = embeddings[target_indices]

    # Créer un nouvel index FAISS pour le sous-ensemble
    dimension = target_embeddings.shape[1]
    target_index = faiss.IndexFlatIP(dimension)
    faiss.normalize_L2(target_embeddings)
    target_index.add(target_embeddings)

    return target_index, target_indices



#### Fonction principale de matching hybride

In [ ]:
def hybrid_matching(query_text, query_type, top_k=5, confidence_threshold=0.7, use_sampled_data=True):
    """
    Fonction principale de matching hybride
    """
    # Charger les modèles et données
    (cv_classifier, offer_classifier, cv_le, offer_le,
     tokenizer, distilbert_model) = load_classification_models()

    (cv_dataset, offer_dataset, cv_index, offer_index,
     cv_embeddings, offer_embeddings, cv_id_to_index,
     offer_id_to_index, sbert_model) = load_matching_data(use_sampled_data=use_sampled_data)

    manual_mapping = create_manual_class_mapping()
    high_conf_cv_classes, high_conf_offer_classes = load_high_confidence_classes(confidence_threshold)

    # Déterminer le type de requête et configurer en conséquence
    if query_type == 'cv':
        classifier = cv_classifier
        le = cv_le
        high_conf_classes = high_conf_cv_classes
        target_data_type = 'offer'
        target_dataset = offer_dataset
        target_index_base = offer_index
        target_embeddings = offer_embeddings
        target_id_to_index = offer_id_to_index
    else:
        classifier = offer_classifier
        le = offer_le
        high_conf_classes = high_conf_offer_classes
        target_data_type = 'cv'
        target_dataset = cv_dataset
        target_index_base = cv_index
        target_embeddings = cv_embeddings
        target_id_to_index = cv_id_to_index

    # Étape 1: Classification de la requête
    predicted_class, confidence_score, is_high_confidence = classify_text(
        query_text, query_type, classifier, le, tokenizer, distilbert_model, confidence_threshold)

    print(f"Classification: {predicted_class} (confiance: {confidence_score:.3f})")

    # Étape 2: Détermination des classes cibles
    if is_high_confidence and predicted_class in high_conf_classes:
        target_classes = get_target_classes(predicted_class, query_type, manual_mapping)
        print(f"Classes cibles ({target_data_type}): {target_classes}")

       # Étape 3: Filtrage par classes cibles
    if is_high_confidence and predicted_class in high_conf_classes:
        target_classes = get_target_classes(predicted_class, query_type, manual_mapping)
        print(f"Classes cibles ({target_data_type}): {target_classes}")

        # Vérifier si les classes cibles existent dans le dataset
        available_categories = target_dataset.iloc[:, 1].unique()
        existing_target_classes = [cls for cls in target_classes if cls in available_categories]

        if not existing_target_classes:
            print(f"Avertissement: Aucune des classes cibles {target_classes} n'existe dans le dataset {target_data_type}")
            print(f"Classes disponibles: {list(available_categories)}")
            # Fallback: utiliser toutes les classes
            target_index = target_index_base
            target_indices = None
        else:
            # Utiliser seulement les classes qui existent
            target_index, target_indices = filter_by_class(
                target_dataset, existing_target_classes, target_id_to_index,
                target_embeddings, target_index_base)
    else:
        print("Confiance insuffisante - utilisation de toutes les classes")
        target_index = target_index_base
        target_indices = None

    # Étape 4: Matching sémantique avec SBERT
    query_embedding = sbert_model.encode([query_text])
    faiss.normalize_L2(query_embedding)

    # Recherche des similarités
    if target_index is not None:
        distances, indices = target_index.search(query_embedding, top_k)

        # Préparation des résultats
        results = []
        for i, (distance, idx) in enumerate(zip(distances[0], indices[0])):
            if idx < 0:  # Index invalide
                continue

            # Si nous avons un sous-ensemble, mapper vers l'index original
            if target_indices is not None:
                original_idx = target_indices[idx]
            else:
                original_idx = idx

            # Récupérer les métadonnées du document
            if target_data_type == 'offer':
                doc_id = target_dataset.iloc[original_idx]['offer_id']
                doc_title = target_dataset.iloc[original_idx]['Job Title_Category']
                doc_field = 'job_title'
            else:
                doc_id = target_dataset.iloc[original_idx]['cv_id']
                doc_title = target_dataset.iloc[original_idx]['Category']
                doc_field = 'category'

            results.append({
                'rank': i + 1,
                'id': doc_id,
                doc_field: doc_title,
                'similarity_score': float(distance)
            })

        return results, predicted_class, confidence_score
    else:
        print("Aucun document trouvé dans les classes cibles")
        return [], predicted_class, confidence_score


def hybrid_matching(query_text, query_type, top_k=5, confidence_threshold=0.7, use_sampled_data=True):
    """
    Fonction principale de matching hybride avec affichage amélioré
    """
    # Charger les modèles et données
    (cv_classifier, offer_classifier, cv_le, offer_le,
     tokenizer, distilbert_model) = load_classification_models()

    (cv_dataset, offer_dataset, cv_index, offer_index,
     cv_embeddings, offer_embeddings, cv_id_to_index,
     offer_id_to_index, sbert_model) = load_matching_data(use_sampled_data=use_sampled_data)

    manual_mapping = create_manual_class_mapping()
    high_conf_cv_classes, high_conf_offer_classes = load_high_confidence_classes(confidence_threshold)

    # Déterminer le type de requête et configurer en conséquence
    if query_type == 'cv':
        classifier = cv_classifier
        le = cv_le
        high_conf_classes = high_conf_cv_classes
        target_data_type = 'offer'
        target_dataset = offer_dataset
        target_index_base = offer_index
        target_embeddings = offer_embeddings
        target_id_to_index = offer_id_to_index
    else:
        classifier = offer_classifier
        le = offer_le
        high_conf_classes = high_conf_offer_classes
        target_data_type = 'cv'
        target_dataset = cv_dataset
        target_index_base = cv_index
        target_embeddings = cv_embeddings
        target_id_to_index = cv_id_to_index

    # Étape 1: Classification de la requête
    predicted_class, confidence_score, is_high_confidence = classify_text(
        query_text, query_type, classifier, le, tokenizer, distilbert_model, confidence_threshold)

    # Afficher le résultat de classification
    display_classification_result(query_text, query_type, predicted_class, confidence_score)

    # Étape 2: Détermination des classes cibles
    if is_high_confidence and predicted_class in high_conf_classes:
        target_classes = get_target_classes(predicted_class, query_type, manual_mapping)
        print(f"   ➡️  Classes cibles recherchées : {target_classes}")

         # Étape 3: Filtrage par classes cibles
    if is_high_confidence and predicted_class in high_conf_classes:
        target_classes = get_target_classes(predicted_class, query_type, manual_mapping)
        print(f"Classes cibles ({target_data_type}): {target_classes}")

        # Vérifier si les classes cibles existent dans le dataset
        available_categories = target_dataset.iloc[:, 1].unique()
        existing_target_classes = [cls for cls in target_classes if cls in available_categories]

        if not existing_target_classes:
            print(f"Avertissement: Aucune des classes cibles {target_classes} n'existe dans le dataset {target_data_type}")
            print(f"Classes disponibles: {list(available_categories)}")
            # Fallback: utiliser toutes les classes
            target_index = target_index_base
            target_indices = None
        else:
            # Utiliser seulement les classes qui existent
            target_index, target_indices = filter_by_class(
                target_dataset, existing_target_classes, target_id_to_index,
                target_embeddings, target_index_base)
    else:
        print("   ⚠️  Confiance insuffisante - recherche dans toutes les classes")
        target_index = target_index_base
        target_indices = None

    # Étape 4: Matching sémantique avec SBERT
    query_embedding = sbert_model.encode([query_text])
    faiss.normalize_L2(query_embedding)

    # Recherche des similarités
    if target_index is not None:
        distances, indices = target_index.search(query_embedding, top_k)

        # Préparation des résultats
        results = []
        for i, (distance, idx) in enumerate(zip(distances[0], indices[0])):
            if idx < 0:  # Index invalide
                continue

            # Si nous avons un sous-ensemble, mapper vers l'index original
            if target_indices is not None:
                original_idx = target_indices[idx]
            else:
                original_idx = idx

            # Récupérer les métadonnées du document
            if target_data_type == 'offer':
                doc_id = target_dataset.iloc[original_idx]['offer_id']
                doc_title = target_dataset.iloc[original_idx]['Job Title_Category']
                doc_field = 'job_title'
            else:
                doc_id = target_dataset.iloc[original_idx]['cv_id']
                doc_title = target_dataset.iloc[original_idx]['Category']
                doc_field = 'category'

            results.append({
                'rank': i + 1,
                'id': doc_id,
                doc_field: doc_title,
                'similarity_score': float(distance)
            })

        # Afficher les résultats de matching
        display_matching_results(results, query_type, predicted_class, confidence_score)

        return results, predicted_class, confidence_score
    else:
        print("\n Aucun document trouvé dans les classes cibles spécifiées")
        print("   Vérification des catégories disponibles...")

        # Afficher les catégories disponibles pour diagnostic
        available_categories = target_dataset.iloc[:, 1].unique()
        print(f"   Catégories disponibles : {list(available_categories)}")

        return [], predicted_class, confidence_score

### AFFICHAGE EVALUATION

In [ ]:
from tabulate import tabulate
def display_classification_result(query_text, query_type, predicted_class, confidence_score):
    """
    Affiche le résultat de la classification de manière claire et précise
    """
    # Extraire un ID à partir du texte (première partie du hash)
    query_id = generate_id(query_text)[:10]

    if query_type == 'cv':
        print(f"\n CLASSIFICATION DU CV [{query_id}...]")
        print(f"   Ce CV est classifié comme : '{predicted_class}'")
    else:
        print(f"\n CLASSIFICATION DE L'OFFRE [{query_id}...]")
        print(f"   Cette offre est classifiée comme : '{predicted_class}'")

    print(f"   Niveau de confiance : {confidence_score*100:.1f}%")

    # Interprétation du niveau de confiance
    if confidence_score >= 0.9:
        confidence_level = "Très élevée"
    elif confidence_score >= 0.7:
        confidence_level = "Élevée"
    elif confidence_score >= 0.5:
        confidence_level = "Modérée"
    else:
        confidence_level = "Faible"

    print(f"   Niveau de confiance : {confidence_level}")

def display_matching_results(results, query_type, predicted_class, confidence_score):
    """
    Affiche les résultats de matching de manière claire et précise
    """
    if not results:
        print("\n  Aucune correspondance trouvée")
        return

    if query_type == 'cv':
        print(f"\n  OFFRES CORRESPONDANTES POUR LE PROFIL '{predicted_class}'")
        print(f"   (Confiance de classification: {confidence_score*100:.1f}%)")
    else:
        print(f"\n CVS CORRESPONDANTS POUR L'OFFRE '{predicted_class}'")
        print(f"   (Confiance de classification: {confidence_score*100:.1f}%)")

    print("   " + "="*60)

    for i, result in enumerate(results[:5]):  # Limiter aux 5 premiers résultats
        if 'job_title' in result:
            profile_info = f"{result['job_title']}"
            result_type = "Offre"
        else:
            profile_info = f"{result['category']}"
            result_type = "CV"

        # Formater le score de similarité
        similarity_score = result['similarity_score']
        if similarity_score >= 0.8:
            similarity_quality = "Excellente correspondance"
        elif similarity_score >= 0.6:
            similarity_quality = "Bonne correspondance"
        elif similarity_score >= 0.4:
            similarity_quality = "Correspondance moyenne"
        else:
            similarity_quality = "Faible correspondance"

        print(f"\n   {i+1}. {result_type} [{result['id'][:8]}...]")
        print(f"      Profil : {profile_info}")
        print(f"      Score de similarité : {similarity_score:.3f}")
        print(f"      Qualité : {similarity_quality}")

    # Afficher le meilleur résultat avec plus de détails
    best_result = results[0]
    if 'job_title' in best_result:
        best_profile = best_result['job_title']
        best_type = "Offre"
    else:
        best_profile = best_result['category']
        best_type = "CV"

    print(f"\n MEILLEURE CORRESPONDANCE")
    print(f"   {best_type} [{best_result['id']}]")
    print(f"   Profil : {best_profile}")
    print(f"   Score de similarité : {best_result['similarity_score']:.3f}")

def display_evaluation_results(evaluation_results):
    """
    Affiche les résultats de l'évaluation de manière claire
    """
    print("\n" + "="*80)
    print("RÉSULTATS DE L'ÉVALUATION DU SYSTÈME")
    print("="*80)

    # Calcul des métriques
    total_tests = len(evaluation_results)
    correct_classifications = sum(1 for r in evaluation_results if r['classification_correct'])
    classification_accuracy = correct_classifications / total_tests
    avg_confidence = np.mean([r['classification_confidence'] for r in evaluation_results])
    avg_matching_score = np.mean([r['top_matching_score'] for r in evaluation_results if r['has_matching_results']])
    success_rate = sum(1 for r in evaluation_results if r['has_matching_results']) / total_tests

    print(f"• Précision de classification : {classification_accuracy*100:.1f}%")
    print(f"• Confiance moyenne : {avg_confidence*100:.1f}%")
    print(f"• Score de matching moyen : {avg_matching_score:.3f}")
    print(f"• Taux de succès (au moins une correspondance) : {success_rate*100:.1f}%")

    # Détail par test
    print(f"\n• DÉTAIL DES TESTS ({total_tests} tests effectués)")
    print("  " + "-"*70)

    for result in evaluation_results:
        status_icon = "✅" if result['classification_correct'] else "❌"
        match_icon = "✓" if result['has_matching_results'] else "✗"

        print(f"  {status_icon} Test {result['test_id']}: {result['query_type'].upper()}")
        print(f"     Attendu: {result['expected_class']}, Prédit: {result['predicted_class']}")
        print(f"     Confiance: {result['classification_confidence']*100:.1f}%")
        print(f"     Matching: {match_icon} (meilleur score: {result['top_matching_score']:.3f})")
        print()


def evaluate_hybrid_system(test_cases, confidence_threshold=0.7, use_sampled_data=True):
    """
    Évalue le système hybride sur un ensemble de tests
    """
    print("ÉVALUATION DU SYSTÈME HYBRIDE")
    print("="*80)

    results = []

    for i, (query_text, query_type, expected_class) in enumerate(test_cases):
        print(f"\nTest {i+1}/{len(test_cases)}: {query_type.upper()}")
        print(f"Texte: {query_text[:100]}...")

        # Exécuter le matching hybride
        matching_results, predicted_class, confidence = hybrid_matching(
            query_text, query_type, top_k=3, confidence_threshold=confidence_threshold, use_sampled_data=use_sampled_data)

        # Évaluer la classification
        classification_correct = (predicted_class == expected_class)
        classification_confidence = confidence

        # Évaluer le matching (simplifié)
        has_results = len(matching_results) > 0
        top_score = matching_results[0]['similarity_score'] if has_results else 0

        results.append({
            'test_id': i + 1,
            'query_type': query_type,
            'expected_class': expected_class,
            'predicted_class': predicted_class,
            'classification_correct': classification_correct,
            'classification_confidence': classification_confidence,
            'has_matching_results': has_results,
            'top_matching_score': top_score
        })

        print(f"Résultat: {'✓' if classification_correct else '✗'} {predicted_class} "
              f"(confiance: {confidence:.3f}, matching: {top_score:.3f})")

    # Calcul des métriques
    total_tests = len(results)
    correct_classifications = sum(1 for r in results if r['classification_correct'])
    classification_accuracy = correct_classifications / total_tests
    avg_confidence = np.mean([r['classification_confidence'] for r in results])
    avg_matching_score = np.mean([r['top_matching_score'] for r in results if r['has_matching_results']])

    print(f"\n{'='*80}")
    print("RÉSULTATS DE L'ÉVALUATION")
    print(f"{'='*80}")
    print(f"Précision de classification: {classification_accuracy:.3f}")
    print(f"Confiance moyenne: {avg_confidence:.3f}")
    print(f"Score de matching moyen: {avg_matching_score:.3f}")

    return results

### SYSTEM MACTHING

In [ ]:
# Exemple d'utilisation
# if __name__ == "__main__":
print("🔍 SYSTÈME DE MATCHING CV-OFFRES AVEC CLASSIFICATION HYBRIDE")
print("="*80)

# Test avec un CV
sample_cv = "Data analyst with 5 years of experience in SQL, Python, and data visualization. Strong analytical skills and experience with Tableau and Power BI."
results, predicted_class, confidence = hybrid_matching(sample_cv, "cv", top_k=5, use_sampled_data=True)

# Test avec une offre
sample_offer = "Looking for a data scientist with machine learning experience and Python programming skills. Knowledge of TensorFlow or PyTorch is a plus."
results, predicted_class, confidence = hybrid_matching(sample_offer, "offer", top_k=5, use_sampled_data=True)

# Évaluation du système
test_cases = [
    ("Data analyst with SQL and Python experience", "cv", "Data Analyst"),
    ("Software developer with Java and Spring framework", "cv", "Software Developer"),
    ("Data scientist position with machine learning requirements", "offer", "Data Scientist"),
    ("Web developer with React and JavaScript", "offer", "Web Developer")
]

evaluation_results = evaluate_hybrid_system(test_cases, use_sampled_data=True)
display_evaluation_results(evaluation_results)

🔍 SYSTÈME DE MATCHING CV-OFFRES AVEC CLASSIFICATION HYBRIDE


Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFDistilBertModel: ['vocab_transform.bias', 'vocab_layer_norm.bias', 'vocab_transform.weight', 'vocab_layer_norm.weight', 'vocab_projector.weight', 'vocab_projector.bias']
- This IS expected if you are initializing TFDistilBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFDistilBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFDistilBertModel for predictions without further training.


Échantillon CVs créé: 1200 éléments
Échantillon offres créé: 800 éléments
Génération des embeddings pour les CV...


Batches:   0%|          | 0/38 [00:00<?, ?it/s]

Génération des embeddings pour les offres...


...

Génération des embeddings pour les offres...


Batches:   0%|          | 0/25 [00:00<?, ?it/s]

Embeddings et index FAISS générés avec succès
Fichiers de métriques non trouvés, utilisation de toutes les classes


Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFDistilBertModel: ['vocab_transform.bias', 'vocab_layer_norm.bias', 'vocab_transform.weight', 'vocab_layer_norm.weight', 'vocab_projector.weight', 'vocab_projector.bias']
- This IS expected if you are initializing TFDistilBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFDistilBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFDistilBertModel for predictions without further training.
Some weights of the PyTorch model were not used w

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 118ms/step

 CLASSIFICATION DE L'OFFRE [8c9c2dbcea...]
   Cette offre est classifiée comme : 'Software Developer'
   Niveau de confiance : 46.9%
   Niveau de confiance : Faible
   ⚠️  Confiance insuffisante - recherche dans toutes les classes

 CVS CORRESPONDANTS POUR L'OFFRE 'Software Developer'
   (Confiance de classification: 46.9%)

   1. CV [5b277303...]
      Profil : Consultant
      Score de similarité : 0.636
      Qualité : Bonne correspondance

   2. CV [08a95ea8...]
      Profil : UI Developer
      Score de similarité : 0.606
      Qualité : Bonne correspondance

   3. CV [d2dfc0ff...]
      Profil : Software Engineer
      Score de similarité : 0.569
      Qualité : Correspondance moyenne

 MEILLEURE CORRESPONDANCE
   CV [5b27730377e7]
   Profil : Consultant
   Score de similarité : 0.636
Résultat: ✗ Software Developer (confiance: 0.469, matching: 0.636)

RÉSULTATS DE L'ÉVALUATION
Précision de classification: 0.500
Confiance moyenne: 0.712
Scor

## EVALUATION valid

In [ ]:
# Classes offres avec F1 > 70%
high_f1_offer_classes = [
    'AI/ML Specialist', 'Data Analyst', 'Data Engineer',
    'Data Scientist', 'DevOps/Cloud', 'HR Professional',
    'Librarian/Archivist','Mechanical Engineer',
    'Other No IT Professionnel', 'Software Developer'
]

# Classes CVs avec F1 > 70%
high_f1_cv_classes = [
    'Big Data Cloud Developer', 'Cyber Security Analyst', 'Data Scientist',
    'Database Administrator', 'Front End Developer', 'Full Stack Developer',
    'Information Security Analyst','Java Developer,Software Developer',
    'Java Full Stack Developer', 'Mobile Developer', 'Network Administrator',
    'Network Engineer', 'Oracle Database Administrator','Other No IT', 'Python Developer',
    'SQL/SQL Server Database Administrator','Security Analyst', 'Senior Database Administrator',
    'Senior Java Full Stack Developer','Senior Python Developer', 'Software Developer', 'Software Engineer',
    'Systems Administrator', 'UI Developer', 'Web Developer'
]

def create_evaluation_samples(cv_dataset, offer_dataset, cv_sample_size=100, offer_sample_size=100):
    """
    Crée des échantillons d'évaluation avec les classes à haut F1-score
    """
    # Filtrer les CVs avec les classes à haut F1-score
    cv_eval_samples = cv_dataset[cv_dataset['Category'].isin(high_f1_cv_classes)]

    # Si nous avons plus d'échantillons que nécessaire, en sélectionner aléatoirement
    if len(cv_eval_samples) > cv_sample_size:
        cv_eval_samples = cv_eval_samples.sample(n=cv_sample_size, random_state=42)

    # Filtrer les offres avec les classes à haut F1-score
    offer_eval_samples = offer_dataset[offer_dataset['Job Title_Category'].isin(high_f1_offer_classes)]

    # Si nous avons plus d'échantillons que nécessaire, en sélectionner aléatoirement
    if len(offer_eval_samples) > offer_sample_size:
        offer_eval_samples = offer_eval_samples.sample(n=offer_sample_size, random_state=42)

    print(f"Échantillon CVs créé: {len(cv_eval_samples)} éléments")
    print(f"Échantillon offres créé: {len(offer_eval_samples)} éléments")

    return cv_eval_samples, offer_eval_samples

def prepare_evaluation_pairs(cv_samples, offer_samples, manual_mapping, num_pairs=50):
    """
    Prépare des paires d'évaluation basées sur le mapping manuel
    """
    evaluation_pairs = []

    # Pour chaque CV, trouver les offres correspondantes selon le mapping
    for _, cv_row in cv_samples.iterrows():
        cv_class = cv_row['Category']
        cv_id = cv_row['cv_id']
        cv_text = cv_row['Resume']

        # Trouver les classes d'offres correspondantes
        if cv_class in manual_mapping:
            target_offer_classes = [manual_mapping[cv_class]]
        else:
            # Si pas de mapping direct, utiliser la classe CV comme fallback
            target_offer_classes = [cv_class]

        # Trouver les offres dans ces classes
        matching_offers = offer_samples[offer_samples['Job Title_Category'].isin(target_offer_classes)]

        if len(matching_offers) > 0:
            # Prendre une offre au hasard parmi les correspondantes
            offer_row = matching_offers.sample(n=1, random_state=42).iloc[0]

            evaluation_pairs.append({
                'cv_id': cv_id,
                'cv_text': cv_text,
                'cv_class': cv_class,
                'offer_id': offer_row['offer_id'],
                'offer_text': offer_row['Job Description'],
                'offer_class': offer_row['Job Title_Category'],
                'expected_match': True  # Ces paires devraient correspondre
            })

            if len(evaluation_pairs) >= num_pairs:
                break

    # Ajouter quelques paires qui ne devraient pas correspondre (negatives)
    negative_pairs = 10
    for _ in range(negative_pairs):
        # Prendre un CV et une offre de classes différentes
        cv_sample = cv_samples.sample(n=1, random_state=42).iloc[0]
        offer_sample = offer_samples.sample(n=1, random_state=42).iloc[0]

        # Vérifier qu'ils ne sont pas dans des classes mappées
        cv_class = cv_sample['Category']
        offer_class = offer_sample['Job Title_Category']

        if (cv_class not in manual_mapping or
            manual_mapping[cv_class] != offer_class):

            evaluation_pairs.append({
                'cv_id': cv_sample['cv_id'],
                'cv_text': cv_sample['Resume'],
                'cv_class': cv_class,
                'offer_id': offer_sample['offer_id'],
                'offer_text': offer_sample['Job Description'],
                'offer_class': offer_class,
                'expected_match': False  # Ces paires ne devraient pas correspondre
            })

    print(f"Paires d'évaluation créées: {len(evaluation_pairs)} (dont {negative_pairs} négatives)")
    return evaluation_pairs


In [ ]:
def evaluate_with_high_f1_classes():
    """
    Évaluation complète du système en utilisant seulement les classes à haut F1-score
    """
    print("🔍 ÉVALUATION AVEC CLASSES HAUT F1-SCORE")
    print("="*80)

    # Charger les données
    cv_dataset, offer_dataset, cv_id_to_index, offer_id_to_index = load_data_sampling()

    # Créer des échantillons d'évaluation
    cv_eval_samples, offer_eval_samples = create_evaluation_samples(
        cv_dataset, offer_dataset, cv_sample_size=210, offer_sample_size=210)

    # Charger le mapping manuel
    manual_mapping = create_manual_class_mapping()

    # Préparer les paires d'évaluation
    evaluation_pairs = prepare_evaluation_pairs(
        cv_eval_samples, offer_eval_samples, manual_mapping, num_pairs=50)

    # Charger les modèles et données pour le matching
    (cv_classifier, offer_classifier, cv_le, offer_le,
     tokenizer, distilbert_model) = load_classification_models()

    (_, _, cv_index, offer_index,
     cv_embeddings, offer_embeddings, _,
     _, sbert_model) = load_matching_data(use_sampled_data=True)

    # Évaluation
    results = []

    for i, pair in enumerate(evaluation_pairs):
        print(f"\n--- Paire {i+1}/{len(evaluation_pairs)} ---")
        print(f"CV: {pair['cv_class']}, Offre: {pair['offer_class']}")
        print(f"Attendu: {'Correspondance' if pair['expected_match'] else 'Non-correspondance'}")

        # Classification du CV
        cv_pred_class, cv_confidence, _ = classify_text(
            pair['cv_text'], 'cv', cv_classifier, cv_le, tokenizer, distilbert_model)

        # Classification de l'offre
        offer_pred_class, offer_confidence, _ = classify_text(
            pair['offer_text'], 'offer', offer_classifier, offer_le, tokenizer, distilbert_model)

        # Vérifier si les classifications sont correctes
        cv_class_correct = (cv_pred_class == pair['cv_class'])
        offer_class_correct = (offer_pred_class == pair['offer_class'])

        # Matching sémantique
        cv_embedding = sbert_model.encode([pair['cv_text']])
        offer_embedding = sbert_model.encode([pair['offer_text']])
        faiss.normalize_L2(cv_embedding)
        faiss.normalize_L2(offer_embedding)

        # Calcul de la similarité cosinus
        similarity = np.dot(cv_embedding, offer_embedding.T)[0][0]

        # Déterminer si c'est une correspondance basée sur le seuil
        match_threshold = 0.6  # Seuil de similarité
        is_match = similarity >= match_threshold

        # Évaluer la performance
        match_correct = (is_match == pair['expected_match'])

        results.append({
            'pair_id': i + 1,
            'cv_class': pair['cv_class'],
            'cv_pred_class': cv_pred_class,
            'cv_confidence': cv_confidence,
            'cv_class_correct': cv_class_correct,
            'offer_class': pair['offer_class'],
            'offer_pred_class': offer_pred_class,
            'offer_confidence': offer_confidence,
            'offer_class_correct': offer_class_correct,
            'similarity': similarity,
            'expected_match': pair['expected_match'],
            'predicted_match': is_match,
            'match_correct': match_correct
        })

        print(f"Résultat: CV={cv_pred_class}({cv_confidence:.3f}), "
              f"Offre={offer_pred_class}({offer_confidence:.3f}), "
              f"Similarité={similarity:.3f}, "
              f"Match={'✓' if is_match else '✗'} ({'✓' if match_correct else '✗'})")

    # Calcul des métriques
    total_pairs = len(results)
    correct_matches = sum(1 for r in results if r['match_correct'])
    match_accuracy = correct_matches / total_pairs

    cv_correct = sum(1 for r in results if r['cv_class_correct'])
    cv_accuracy = cv_correct / total_pairs

    offer_correct = sum(1 for r in results if r['offer_class_correct'])
    offer_accuracy = offer_correct / total_pairs

    avg_similarity = np.mean([r['similarity'] for r in results])

    # Séparer les paires positives et négatives
    positive_pairs = [r for r in results if r['expected_match']]
    negative_pairs = [r for r in results if not r['expected_match']]

    # Métriques pour les paires positives
    if positive_pairs:
        true_positives = sum(1 for r in positive_pairs if r['predicted_match'])
        false_negatives = len(positive_pairs) - true_positives
        recall = true_positives / len(positive_pairs)
        avg_similarity_positive = np.mean([r['similarity'] for r in positive_pairs])
    else:
        recall = 0
        avg_similarity_positive = 0

    # Métriques pour les paires négatives
    if negative_pairs:
        true_negatives = sum(1 for r in negative_pairs if not r['predicted_match'])
        false_positives = len(negative_pairs) - true_negatives
        specificity = true_negatives / len(negative_pairs)
        avg_similarity_negative = np.mean([r['similarity'] for r in negative_pairs])
    else:
        specificity = 0
        avg_similarity_negative = 0

    # Affichage des résultats
    print(f"\n{'='*80}")
    print("📊 RÉSULTATS DE L'ÉVALUATION")
    print(f"{'='*80}")
    print(f"Précision globale des matching: {match_accuracy*100:.1f}%")
    print(f"Précision classification CVs: {cv_accuracy*100:.1f}%")
    print(f"Précision classification offres: {offer_accuracy*100:.1f}%")
    print(f"Similarité moyenne: {avg_similarity:.3f}")
    print(f"Rappel (paires positives): {recall*100:.1f}%")
    print(f"Spécificité (paires négatives): {specificity*100:.1f}%")
    print(f"Similarité moyenne (paires positives): {avg_similarity_positive:.3f}")
    print(f"Similarité moyenne (paires négatives): {avg_similarity_negative:.3f}")

    # Matrice de confusion
    print(f"\nMatrice de confusion:")
    print(f"Vrais positifs: {true_positives if positive_pairs else 0}")
    print(f"Faux négatifs: {false_negatives if positive_pairs else 0}")
    print(f"Vrais négatifs: {true_negatives if negative_pairs else 0}")
    print(f"Faux positifs: {false_positives if negative_pairs else 0}")

    return results

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## evaluation

In [ ]:
# Exécuter l'évaluation
# if __name__ == "__main__":
# Évaluation avec les classes à haut F1-score
evaluation_results = evaluate_with_high_f1_classes()

# Sauvegarder les résultats
evaluation_df = pd.DataFrame(evaluation_results)
evaluation_df.to_csv('/content/drive/MyDrive/Colab Notebooks/Projet MASTER1/evaluation_results.csv', index=False)
print("Résultats d'évaluation sauvegardés")

# Analyse détaillée par classe
print(f"\n{'='*80}")
print("ANALYSE PAR CLASSE")
print(f"{'='*80}")

# Analyse des CVs
cv_class_performance = evaluation_df.groupby('cv_class').agg({
    'cv_class_correct': 'mean',
    'similarity': 'mean',
    'pair_id': 'count'
}).rename(columns={
    'cv_class_correct': 'accuracy',
    'pair_id': 'count'
}).round(3)

print("Performance par classe de CV:")
print(cv_class_performance)

# Analyse des offres
offer_class_performance = evaluation_df.groupby('offer_class').agg({
    'offer_class_correct': 'mean',
    'similarity': 'mean',
    'pair_id': 'count'
}).rename(columns={
    'offer_class_correct': 'accuracy',
    'pair_id': 'count'
}).round(3)

print("\nPerformance par classe d'offre:")
print(offer_class_performance)

🔍 ÉVALUATION AVEC CLASSES HAUT F1-SCORE
Échantillon CVs créé: 1200 éléments
Échantillon offres créé: 800 éléments
Échantillon CVs créé: 200 éléments
Échantillon offres créé: 200 éléments
Paires d'évaluation créées: 55 (dont 10 négatives)


Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFDistilBertModel: ['vocab_transform.bias', 'vocab_layer_norm.bias', 'vocab_transform.weight', 'vocab_layer_norm.weight', 'vocab_projector.weight', 'vocab_projector.bias']
- This IS expected if you are initializing TFDistilBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
All the weights of TFDistilBertModel were initialized from the PyTorch model.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFDistilBertModel for predictions without further training.


Échantillon CVs créé: 1200 éléments
Échantillon offres créé: 800 éléments
Génération des embeddings pour les CV...


Batches:   0%|          | 0/38 [00:00<?, ?it/s]

Génération des embeddings pour les offres...


Batches:   0%|          | 0/25 [00:00<?, ?it/s]

Embeddings et index FAISS générés avec succès

--- Paire 1/55 ---
CV: Big Data Cloud Developer, Offre: Data Engineer
Attendu: Correspondance
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 683ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step
Résultat: CV=Big Data Cloud Developer(0.976), Offre=Data Analyst(0.755), Similarité=0.619, Match=✓ (✓)

--- Paire 2/55 ---
CV: Data Scientist, Offre: Data Scientist
Attendu: Correspondance
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
Résultat: CV=Data Scientist(0.981), Offre=Data Scientist(0.624), Similarité=0.237, Match=✗ (✗)

--- Paire 3/55 ---
CV: Other No IT, Offre: Other No IT Professionnel
Attendu: Correspondance
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 435ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step
Résultat: CV=Other No IT(0.970), Offre=Other No IT Professionnel(0.992), Similarité=0.425, Match=✗ (✗)

--- Paire 4/55 ---
CV: Big Data Cloud Developer, Offre: Data Engineer
Attendu: Correspondance
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 50ms/step
1/1 ━━━━━━━━━

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_evaluation_results(results):
    """
    Crée des visualisations pour les résultats d'évaluation
    """
    # Convertir en DataFrame
    df = pd.DataFrame(results)

    # 1. Distribution des similarités
    plt.figure(figsize=(12, 5))

    plt.subplot(1, 2, 1)
    sns.histplot(data=df, x='similarity', hue='expected_match', multiple='stack', bins=20)
    plt.title('Distribution des similarités par type de paire')
    plt.axvline(x=0.6, color='r', linestyle='--', label='Seuil de matching (0.6)')
    plt.legend()

    plt.subplot(1, 2, 2)
    sns.boxplot(data=df, x='expected_match', y='similarity')
    plt.title('Similarité par type de paire')

    plt.tight_layout()
    plt.show()

    # 2. Performance des classes
    plt.figure(figsize=(15, 6))

    # Performance classification CVs
    cv_perf = df.groupby('cv_class')['cv_class_correct'].mean().sort_values(ascending=False)
    plt.subplot(1, 2, 1)
    cv_perf.plot(kind='bar')
    plt.title('Précision de classification par classe (CVs)')
    plt.xticks(rotation=45, ha='right')
    plt.ylim(0, 1)

    # Performance classification offres
    offer_perf = df.groupby('offer_class')['offer_class_correct'].mean().sort_values(ascending=False)
    plt.subplot(1, 2, 2)
    offer_perf.plot(kind='bar')
    plt.title('Précision de classification par classe (Offres)')
    plt.xticks(rotation=45, ha='right')
    plt.ylim(0, 1)

    plt.tight_layout()
    plt.show()

    # 3. Matrice de confusion
    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

    y_true = [1 if r['expected_match'] else 0 for r in results]
    y_pred = [1 if r['predicted_match'] else 0 for r in results]

    cm = confusion_matrix(y_true, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Non-match', 'Match'])

    plt.figure(figsize=(8, 6))
    disp.plot(cmap='Blues')
    plt.title('Matrice de confusion du matching')
    plt.show()

# Utilisation
plot_evaluation_results(evaluation_results)